# Numerical check of off-diagonal induced metric
This notebook builds a small sanity test:
- Pick small-dimension data ($N=2$) for $\gamma, \vec{a}, \vec{b}, \theta$.
- Define a simple quadratic loss so gradients are easy to compute.
- Compute the induced metric via the **definition**: $g = J^\top h J$, then invert numerically.
- Compute $g^{-1}$ using the **rank-2 Sherman–Morrison–Woodbury expression** from the note.
- Compare the two matrices and the action on the gradient; report residual norms to confirm the formula.


## Mathematical setup
- Embed the parameter vector via $\phi(\theta) = (\theta,\mathcal{L}(\theta))$ so the Jacobian is $J = \begin{pmatrix} I_N \\ l^\top \end{pmatrix}$ where $l_i = \partial_i \mathcal{L}$.
- The ambient bilinear form is  $\displaystyle h = \begin{pmatrix} \gamma & a \\ b^\top & 1 \end{pmatrix}$ and the induced metric is $g = J^\top h J$.
- The goal is to compare the numerical inverse of this $g$ with the closed-form inverse obtained by treating $g = \gamma + a l^\top + l(b + l)^\top$ as a rank-2 update and applying Sherman–Morrison–Woodbury.
- The comparisons below look at both the full matrix difference and the action on the geometric direction to make sure the formula replicates the pullback definition.


In [6]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=5, suppress=True)


def make_spd(n: int, rng: np.random.Generator) -> np.ndarray:
    """Construct a random symmetric positive definite matrix for the ambient block."""
    M = rng.normal(size=(n, n))
    # M^T M is positive semi-definite; adding 0.5 I ensures strict positive definiteness.
    return M.T @ M + 0.5 * np.eye(n)


def quadratic_loss(theta: np.ndarray, Q: np.ndarray, c: np.ndarray) -> tuple[float, np.ndarray]:
    """Return the value and gradient of 0.5*theta^T Q theta + c^T theta."""
    loss = 0.5 * theta.T @ Q @ theta + c.T @ theta
    grad = Q @ theta + c
    return loss, grad


def induced_metric_def(gamma: np.ndarray, a: np.ndarray, b: np.ndarray, l: np.ndarray) -> np.ndarray:
    """Compute g = J^T h J directly using the ambient bilinear form."""
    n = gamma.shape[0]
    # Jacobian of phi: top part is identity on theta, bottom is the gradient direction l^T.
    J = np.vstack([np.eye(n), l[np.newaxis, :]])  # shape (n+1, n)
    h = np.block([[gamma, a[:, None]], [b[None, :], np.array([[1.0]])]])
    return J.T @ h @ J


def induced_metric_inv_closed_form(gamma: np.ndarray, a: np.ndarray, b: np.ndarray, l: np.ndarray) -> np.ndarray:
    """Use the rank-2 Sherman–Morrison–Woodbury inverse from the notes."""
    gamma_inv = np.linalg.inv(gamma)
    U = np.column_stack([a, l])            # shape (n, 2)
    V = np.column_stack([l, b + l])        # shape (n, 2)
    middle = np.eye(2) + V.T @ gamma_inv @ U
    # SMW: (gamma + U V^T)^{-1} = gamma_inv - gamma_inv U (I + V^T gamma_inv U)^{-1} V^T gamma_inv.
    return gamma_inv - gamma_inv @ U @ np.linalg.inv(middle) @ V.T @ gamma_inv


# Run a few random checks in dimension N=2 to keep the matrices small and visible.
results = pd.DataFrame(columns=["trial", "res_matrix", "res_action", "det_g_def", "cond_g_def"])
results.set_index("trial", inplace=True)
rng = np.random.default_rng(0)
for trial in range(10):
    n = 2
    gamma = make_spd(n, rng)
    a = rng.normal(size=n)
    b = rng.normal(size=n)
    Q = make_spd(n, rng)
    c = rng.normal(size=n)
    theta = rng.normal(size=n)
    _, l = quadratic_loss(theta, Q, c)

    g_def = induced_metric_def(gamma, a, b, l)
    g_inv_def = np.linalg.inv(g_def)
    g_inv_smw = induced_metric_inv_closed_form(gamma, a, b, l)

    # Compare the full matrices and their action on the gradient direction l.
    res_matrix = np.linalg.norm(g_inv_def - g_inv_smw)
    res_action = np.linalg.norm(g_inv_def @ l - g_inv_smw @ l)

    det_g_def = np.linalg.det(g_def)
    cond_g_def = np.linalg.cond(g_def)

    results.loc[trial] = [res_matrix, res_action, det_g_def, cond_g_def]
    
results


,res_matrix,res_action,det_g_def,cond_g_def
trial,,,,
0,2.267819e-15,4.782987e-15,10.894981,51.909233
1,4.819410e-16,1.200890e-15,9.064703,9.820842
2,6.262681e-15,1.734224e-14,30.575834,91.123190
3,1.209837e-16,1.570092e-16,16.612834,5.385928
4,5.239007e-15,4.632238e-14,354.014317,12.560842
5,3.734131e-16,1.460270e-15,68.161742,23.769441
6,1.457271e-16,5.117875e-16,73.451574,3.096818
7,1.045749e-15,1.023575e-15,-2.652748,1.977307
8,4.710277e-16,7.447602e-16,11.968191,21.939919
